## Full implementation process of fine tuning flan-t5 model code ran on Google Colab.

In [ ]:
!pip install -q --no-cache-dir \
  "transformers==4.46.3" \
  "tokenizers==0.20.1" \
  "accelerate==0.34.2" \
  "peft==0.13.2" \
  sentencepiece safetensors

!pip install datasets evaluate rouge-score torch tensorboard

In [ ]:
import nltk
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
nltk.download("punkt", quiet=True)
nltk.download('punkt_tab')

# 1. Data Loading and Preparation

# Load BioLaySumm dataset
dataset = load_dataset("BioLaySumm/BioLaySumm2025-LaymanRRG-opensource-track")

# Dataset preparation clean up
def clean_up(example):
    src = (example["radiology_report"] or "").strip()
    tgt = (example["layman_report"] or "").strip()
    return len(src) > 0 and len(tgt) > 0

clean_dataset = dataset.filter(clean_up)

# Apply clean up on selected range of required datasets. 
subset_train = clean_dataset["train"].shuffle(seed=42).select(range(100000))
subset_val   = clean_dataset["validation"].select(range(8000))


2. Data Formatting and Preprocessing

# Model + tokenizer
MODEL_NAME = "google/flan-t5-small"
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# 4. Preprocessing 
PREFIX = "Summarize this radiology report for a layperson: "
MAX_INPUT_LEN = 256
MAX_TARGET_LEN = 128

def preprocess_function(batch):
    inputs = [PREFIX + x for x in batch["radiology_report"]]
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding="longest",          # pad dynamically, not to fixed length
    )

    labels = tokenizer(
        text_target=batch["layman_report"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding="longest",
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

cols_to_keep = ["input_ids", "attention_mask", "labels"]

tokenized_train = subset_train.map(
    preprocess_function,
    batched=True,
    remove_columns=[c for c in subset_train.column_names if c not in cols_to_keep],
)
tokenized_val = subset_val.map(
    preprocess_function,
    batched=True,
    remove_columns=[c for c in subset_val.column_names if c not in cols_to_keep],
)
print(first["labels"][:20])


In [ ]:
from google.colab import drive
drive.mount('/content/drive')  # authorize access

In [ ]:
# Metric
metric = evaluate.load("rouge")

def compute_metrics(eval_preds):
   preds, labels = eval_preds

   # decode preds and labels
   labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
   decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
   decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

   # rougeLSum expects newline after each sentence
   decoded_preds = ["\n".join(nltk.sent_tokenize(pred.strip())) for pred in decoded_preds]
   decoded_labels = ["\n".join(nltk.sent_tokenize(label.strip())) for label in decoded_labels]

   result = metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

   return result

# Training arguments
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/saved_models/",
    eval_strategy="epoch",
    learning_rate=2e-4, # smaller lr → more stable
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    num_train_epochs=3,
    logging_steps=10,
    predict_with_generate=True,
    push_to_hub=False,
    report_to="none",
)

model = model.to("cuda").float()

# Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Train
trainer.train()
print(trainer.state.log_history[-10:])

In [ ]:
# Save final model and tokenizer
SAVED_PATH = "/content/drive/MyDrive/saved_models/final"

final_dir = SAVED_PATH
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"Final model and tokenizer saved to {final_dir}")

In [ ]:
# Generate Training Loss Curve

import matplotlib.pyplot as plt

# Extract loss values from training logs
logs = trainer.state.log_history
train_steps = [entry["step"] for entry in logs if "loss" in entry]
train_loss = [entry["loss"] for entry in logs if "loss" in entry]

# Plot training loss curve
plt.figure(figsize=(7,4))
plt.plot(train_steps, train_loss, label="Training Loss")
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title("FLAN-T5 Training Loss Curve")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Generate one example result from training
def generate_lay_summary(radiology_report):
    val = PREFIX + radiology_report
    inputs = tokenizer(val, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, num_beams=2, max_new_tokens=128)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

sample = subset_val[0]
print("\n--- Example Generation ---")
print("Report:\n", sample["radiology_report"][:300], "...\n")
print("Ground Truth summary:\n", sample["layman_report"], "\n")
print("Model summary:\n", generate_lay_summary(sample["radiology_report"]))

In [ ]:
# Predict

from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch

# Load model and tokenizer
SAVED_PATH = "/content/drive/MyDrive/saved_models/final"

model = T5ForConditionalGeneration.from_pretrained(SAVED_PATH, local_files_only=True)
tokenizer = T5Tokenizer.from_pretrained(SAVED_PATH, local_files_only=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# Parameters and prompt
PROMPT_PREFIX = "Summarize this radiology report for a layperson: "
MAX_INPUT_LEN = 256        # same as training
MAX_NEW_TOKENS = 128       # same as MAX_TARGET_LEN
NUM_BEAMS = 4              # beam search for higher quality generation

# Select validation/test subset
subset_test = dataset["validation"]

results = []  # to store (input, ground truth, prediction)

# Generate summaries
for ex in subset_test:
    src_text = ex["radiology_report"]
    tgt_text = ex["layman_report"]

    # build the full input prompt
    input_text = PROMPT_PREFIX + src_text

    # tokenize input
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=MAX_INPUT_LEN,   # consistent with training
        truncation=True,
    ).to(device)

    # generate prediction
    with torch.no_grad():
        pred_ids = model.generate(
            **inputs,
            num_beams=NUM_BEAMS,
            max_new_tokens=MAX_NEW_TOKENS,  # controls output length
        )

    # decode prediction to readable text
    pred_text = tokenizer.decode(pred_ids[0], skip_special_tokens=True)

    # store for evaluation/printing
    results.append((src_text, tgt_text, pred_text))

# Display first few examples
for i, (inp, gt, pred) in enumerate(results[:5]):
    print(f"\n--- Example {i+1} ---")
    print("INPUT:", inp[:250], "...")
    print("GROUND TRUTH:", gt)
    print("MODEL OUTPUT:", pred)